<a href="https://colab.research.google.com/github/NisalJayaweera/EN3160-Fitting-and-Alignment/blob/main/notebooks/Q4_PCB_SIFT_Alignment_GitHub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EN3160 — Assignment 02 | Question 4
## PCB image registration and difference analysis using SIFT + RANSAC

**Reference:** `AECC1.jpg` · **Test images:** `AECC2.jpg`, `AECC3.jpg`, `AECC4.jpg`, `AAAA1.jpg`, `ADAD1.jpg`

This notebook implements all parts **4(a)–4(e)** of the assignment. It downloads the six original PCB photographs from the `main/images/` folder of the GitHub repository, estimates projective mappings into the template coordinate system, and visualizes the differences.

**Reproducibility:** Run all cells in order in Google Colab or a local Jupyter environment with internet access for the initial download. All figures and a metrics CSV are saved to `results/q4/`.

### Method and geometry

1. **4(a):** Detect SIFT keypoints and 128-dimensional descriptors in both grayscale images.
2. **4(b):** Find tentative matches with `cv2.BFMatcher(cv2.NORM_L2)` and Lowe's ratio test; estimate a homography with `cv2.findHomography(..., cv2.RANSAC, ...)`. Draw inlier correspondences.
3. **4(c):** Warp each *test image into the template frame* with `cv2.warpPerspective`. Warp an all-ones mask too, so invalid border pixels are excluded from comparison.
4. **4(d):** Repeat for all five test photographs, show and save each difference image.
5. **4(e):** Discuss why tall or changed components (USB Type-B connector, capacitors, reset switch) produce large residuals.

For corresponding points in homogeneous coordinates, the estimated projective transformation is

\[
\lambda\begin{bmatrix}x_t\\y_t\\1\end{bmatrix}
=H\begin{bmatrix}x_s\\y_s\\1\end{bmatrix},
\qquad H\in\mathbb{R}^{3\times3}.
\]

Here \(s\) is the **source/test** image and \(t\) is the **template** image. RANSAC estimates \(H\) robustly despite mismatched feature pairs. The difference shown is the absolute intensity residual \(D(x,y)=|I_t(x,y)-I_{s\to t}(x,y)|\), restricted to the warped image's valid overlap.

In [ ]:
from pathlib import Path
import urllib.request
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("OpenCV:", cv2.__version__)
assert hasattr(cv2, "SIFT_create"), "Install a version of OpenCV with SIFT support."

REPO = "NisalJayaweera/EN3160-Fitting-and-Alignment"
BRANCH = "main"
IMAGE_DIR = Path("images")
RESULT_DIR = Path("results/q4")
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE_NAME = "AECC1.jpg"
TEST_NAMES = ["AECC2.jpg", "AECC3.jpg", "AECC4.jpg", "AAAA1.jpg", "ADAD1.jpg"]
ALL_NAMES = [TEMPLATE_NAME] + TEST_NAMES

def ensure_image(name):
    destination = IMAGE_DIR / name
    url = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/images/{name}"
    if not destination.is_file() or destination.stat().st_size == 0:
        print("Downloading", url)
        try:
            urllib.request.urlretrieve(url, destination)
        except Exception as exc:
            raise RuntimeError(f"Could not download {name} from GitHub: {url}") from exc
    img = cv2.imread(str(destination), cv2.IMREAD_COLOR)
    if img is None:
        destination.unlink(missing_ok=True)
        raise RuntimeError(f"{destination} is not a readable image. Delete any stale local copy and rerun.")
    return img

images = {name: ensure_image(name) for name in ALL_NAMES}
for name, img in images.items():
    print(f"{name:12s}: {img.shape[1]} x {img.shape[0]}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, name in zip(axes.flat, ALL_NAMES):
    ax.imshow(cv2.cvtColor(images[name], cv2.COLOR_BGR2RGB))
    ax.set_title(name + (" (template)" if name == TEMPLATE_NAME else ""))
    ax.axis("off")
plt.tight_layout()
plt.savefig(RESULT_DIR / "q4_input_images.png", dpi=150, bbox_inches="tight")
plt.show()

### 4(a) SIFT features

SIFT extracts local scale- and rotation-invariant descriptors. To avoid excessive memory consumption on large photographs, feature extraction is performed on resized copies (maximum side `1400` pixels). All displayed alignments and difference images use this same common working resolution for consistency; the homography is therefore estimated in the working-image coordinates.

In [ ]:
MAX_SIDE = 1400
def resize_for_analysis(image, max_side=MAX_SIDE):
    h, w = image.shape[:2]
    scale = min(1.0, max_side / max(h, w))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (round(w*scale), round(h*scale)), interpolation=cv2.INTER_AREA)

working = {name: resize_for_analysis(im) for name, im in images.items()}
grays = {name: cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) for name, im in working.items()}
sift = cv2.SIFT_create(nfeatures=4500)
features = {}
for name in ALL_NAMES:
    kp, desc = sift.detectAndCompute(grays[name], None)
    if desc is None or len(kp) < 4:
        raise RuntimeError(f"Insufficient SIFT features in {name}.")
    features[name] = (kp, desc)
    print(f"{name:12s}: {len(kp)} SIFT keypoints")

fig, ax = plt.subplots(figsize=(12, 7))
marked = cv2.drawKeypoints(working[TEMPLATE_NAME], features[TEMPLATE_NAME][0], None,
                           flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
ax.imshow(cv2.cvtColor(marked, cv2.COLOR_BGR2RGB))
ax.set_title(f"SIFT keypoints — {TEMPLATE_NAME}")
ax.axis("off")
plt.tight_layout()
plt.savefig(RESULT_DIR / "q4_template_sift_keypoints.png", dpi=150, bbox_inches="tight")
plt.show()

### 4(b) Feature matching and RANSAC homography

`knnMatch(k=2)` returns the two nearest descriptor matches. The ratio test rejects ambiguous descriptors. RANSAC then removes geometrically inconsistent matches, yielding a homography supported by inliers. A valid homography requires at least four correspondences; here we also enforce a modest inlier count and check the projected quadrilateral for degeneracy.

In [ ]:
RATIO = 0.75
RANSAC_REPROJECTION_THRESHOLD = 4.0
matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)

def align_and_compare(name):
    ref_name = TEMPLATE_NAME
    kp_src, des_src = features[name]
    kp_ref, des_ref = features[ref_name]

    pairs = matcher.knnMatch(des_src, des_ref, k=2)
    good = [m for pair in pairs if len(pair) == 2 for m, n in [pair] if m.distance < RATIO*n.distance]
    if len(good) < 4:
        raise RuntimeError(f"{name}: only {len(good)} ratio-test matches; need at least 4.")

    src_xy = np.float32([kp_src[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst_xy = np.float32([kp_ref[m.trainIdx].pt for m in good]).reshape(-1,1,2)
    H, mask = cv2.findHomography(src_xy, dst_xy, cv2.RANSAC,
                                ransacReprojThreshold=RANSAC_REPROJECTION_THRESHOLD,
                                maxIters=5000, confidence=0.995)
    if H is None or mask is None:
        raise RuntimeError(f"{name}: homography estimation failed.")
    inlier_mask = mask.ravel().astype(bool)
    n_inliers = int(inlier_mask.sum())
    if n_inliers < 8:
        raise RuntimeError(f"{name}: only {n_inliers} RANSAC inliers; registration is unreliable.")
    # Assess geometric consistency before interpreting pixel differences.
    projected = cv2.perspectiveTransform(src_xy, H).reshape(-1, 2)
    errors = np.linalg.norm(projected - dst_xy.reshape(-1, 2), axis=1)
    median_inlier_error = float(np.median(errors[inlier_mask]))
    reliable = (n_inliers >= 30 and n_inliers / len(good) >= 0.20
                and median_inlier_error <= 3.0)


    src = working[name]
    ref = working[ref_name]
    h, w = ref.shape[:2]
    aligned = cv2.warpPerspective(src, H, (w, h))
    overlap = cv2.warpPerspective(np.full(src.shape[:2], 255, np.uint8), H, (w, h),
                                  flags=cv2.INTER_NEAREST) > 0
    # Avoid edge artifacts at the warped support boundary
    overlap = cv2.erode(overlap.astype(np.uint8), np.ones((5,5),np.uint8), iterations=1).astype(bool)
    if overlap.sum() == 0:
        raise RuntimeError(f"{name}: no overlap with the template.")

    difference = cv2.absdiff(ref, aligned)
    difference[~overlap] = 0
    diff_gray = cv2.cvtColor(difference, cv2.COLOR_BGR2GRAY)
    mae = float(np.mean(difference[overlap]))
    gray_mae = float(np.mean(diff_gray[overlap]))
    overlap_fraction = float(np.mean(overlap))

    # For visualization, show only inlier matches; cap to 70 for readability
    inlier_matches = [m for m, ok in zip(good, inlier_mask) if ok]
    shown = sorted(inlier_matches, key=lambda m: m.distance)[:70]
    match_image = cv2.drawMatches(src, kp_src, ref, kp_ref, shown, None,
                                  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    heat = cv2.applyColorMap(diff_gray, cv2.COLORMAP_JET)
    heat[~overlap] = 0
    return dict(name=name, H=H, matches=len(good), inliers=n_inliers,
                inlier_ratio=n_inliers/len(good), median_inlier_error=median_inlier_error,
                reliable=reliable, aligned=aligned,
                overlap=overlap, difference=difference, diff_gray=diff_gray,
                heat=heat, match_image=match_image, mae=mae,
                gray_mae=gray_mae, overlap_fraction=overlap_fraction)

### 4(c–d) Register all PCB images and visualize differences

For each test image, the output shows the matched SIFT features, the template, the warped test photograph, the absolute difference, and a heatmap. **Black areas outside the overlap are invalid pixels, not perfect matches.** Residual brightness can arise from true PCB changes, lighting/exposure, parallax, occlusion, and imperfect registration.

In [ ]:
results = {}
metrics = []
for name in TEST_NAMES:
    print(f"Processing {name} ...")
    try:
        result = align_and_compare(name)
    except Exception as exc:
        print(f"WARNING: {name} failed: {exc}")
        continue
    results[name] = result
    if not result["reliable"]:
        print(f"  CAUTION: weak registration for {name}; inspect matches and warped image.")
    metrics.append({
        "image": name,
        "ratio_test_matches": result["matches"],
        "ransac_inliers": result["inliers"],
        "inlier_ratio": result["inlier_ratio"],
        "median_inlier_reprojection_error_px": result["median_inlier_error"],
        "registration_quality": "strong" if result["reliable"] else "weak / review",
        "overlap_fraction": result["overlap_fraction"],
        "color_MAE_0_to_255": result["mae"],
        "gray_MAE_0_to_255": result["gray_mae"],
    })

    stem = Path(name).stem
    cv2.imwrite(str(RESULT_DIR / f"{stem}_aligned.jpg"), result["aligned"])
    cv2.imwrite(str(RESULT_DIR / f"{stem}_difference.png"), result["difference"])
    cv2.imwrite(str(RESULT_DIR / f"{stem}_difference_heatmap.png"), result["heat"])
    cv2.imwrite(str(RESULT_DIR / f"{stem}_inlier_matches.jpg"), result["match_image"])

    fig, ax = plt.subplots(figsize=(15, 5))
    ax.imshow(cv2.cvtColor(result["match_image"], cv2.COLOR_BGR2RGB))
    ax.set_title(f"{name}: SIFT inlier matches ({result['inliers']} / {result['matches']} ratio-test matches)")
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{stem}_matches_figure.png", dpi=120, bbox_inches="tight")
    plt.show()

    fig, axs = plt.subplots(1, 4, figsize=(19, 5))
    panels = [
        (working[TEMPLATE_NAME], "Template AECC1"),
        (result["aligned"], f"{stem} warped to template"),
        (result["difference"], "Absolute difference"),
        (result["heat"], "Difference heatmap"),
    ]
    for ax, (img, title) in zip(axs, panels):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(f"{name} | grayscale MAE = {result['gray_mae']:.2f} / 255 | overlap = {result['overlap_fraction']:.1%}")
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{stem}_comparison.png", dpi=140, bbox_inches="tight")
    plt.show()

if len(results) != len(TEST_NAMES):
    raise RuntimeError(f"Only {len(results)} of {len(TEST_NAMES)} PCB images aligned successfully; review warnings above before submission.")
else:
    print("Successfully processed all five test PCB images.")

In [ ]:
metrics_df = pd.DataFrame(metrics)
if metrics_df.empty:
    raise RuntimeError("No successful alignments. Check the images and matching parameters.")
display(metrics_df.round(3))
metrics_df.to_csv(RESULT_DIR / "q4_alignment_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(metrics_df["image"], metrics_df["gray_MAE_0_to_255"])
ax.set_ylabel("Mean absolute grayscale difference (0–255)")
ax.set_title("PCB differences after RANSAC-based alignment")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / "q4_difference_summary.png", dpi=150, bbox_inches="tight")
plt.show()

### 4(e) Discussion — why are some component regions different?

**USB Type-B connector, capacitors, and reset switch:** These are raised three-dimensional components rather than points on a single flat PCB plane. A single homography can align the approximately planar board surface but cannot simultaneously correct the depth-dependent apparent displacement of tall components (parallax). Their sides, shadows, highlights, and partial occlusions can therefore produce strong residuals even if the component is physically unchanged. A component may also genuinely differ in shape, placement, or presence across PCB variants.

**Photometric differences:** Specular metallic connectors, shiny capacitor surfaces, changing light direction, camera exposure, and white balance can increase raw pixel differences. Pixel subtraction is not invariant to illumination.

**Residual registration errors:** Small reprojection errors are amplified near sharp edges, text, solder pads, and component boundaries. Lens distortion and the fact that the board is not perfectly planar can prevent perfect pixel-level alignment.

**Registration quality:** The notebook flags registrations with fewer than 30 inliers, under 20% inlier ratio, or median inlier reprojection error above 3 pixels. These heuristic checks are not proof of correct registration. In particular, inspect the matches and aligned image before treating a large residual as a PCB defect.

**Interpretation of the results:** The RANSAC inlier count and ratio indicate how many SIFT correspondences support a consistent mapping; they are *not* direct measures of PCB correctness. The MAE measures photometric mismatch over valid overlap, not solely manufacturing defects. Examine the saved alignment and difference maps together before concluding that a high-residual region represents a missing or damaged component.

**Possible improvements:** Use a board-only ROI, local illumination normalization, geometric alignment checks, component-level masks, and (when available) camera calibration. A planar homography is appropriate for the board surface, but not for all raised components.

**Limitations:** This experiment uses SIFT + RANSAC for global registration and absolute image differences for visualization. It is not a calibrated industrial defect detector; its outputs depend on viewpoint, lighting, image resolution, and overlap.

**Dataset-specific observation:** AECC2, AECC3 and AECC4 registered with smaller pixel residuals than AAAA1 and ADAD1 in the previous execution. Examine the latter images visually before attributing differences to defects: viewpoint, board configuration, and lower correspondence consensus may dominate the residual. These are descriptive comparisons, not ground-truth defect labels.


### Reproducibility and submission notes

- GitHub input paths: `https://raw.githubusercontent.com/NisalJayaweera/EN3160-Fitting-and-Alignment/main/images/<filename>`.
- Outputs: `results/q4/` (comparison figures, match visualizations, aligned images, difference images, and CSV).
- Report: include representative feature matches, all five difference maps, and a concise discussion of raised-component parallax versus true defects.
- For full reproducibility, commit this notebook under `notebooks/Q4_PCB_SIFT_Alignment_GitHub.ipynb`. Generated figures may be included in the repository if desired.